In [1]:
from pathlib import Path
from PIL import Image
import copy
import cv2
import csv
import random
import numpy as np
from minerva.data.readers.png_reader import PNGReader
from minerva.data.readers.csv_reader import CSVReader
from minerva.data.readers.patched_array_reader import NumpyArrayReader
from minerva.data.data_modules.base import MinervaDataModule

from minerva.transforms.transform import _Transform, Identity
from minerva.transforms.random_transform import _RandomSyncedTransform, RandomFlip, RandomGrayScale, RandomSolarize, RandomRotation
from minerva.data.datasets.base import SimpleDataset

from minerva.utils.tensor import to_tensor

from minerva.models.ssl.byol import BYOL
from minerva.models.nets.image.deeplabv3 import DeepLabV3

from torchvision import transforms
import lightning as L
import torch
from torchmetrics import JaccardIndex
import matplotlib.pyplot as plt
import torchvision.models as models
from torch.utils.data import DataLoader
from torchmetrics.classification import MulticlassJaccardIndex
import torchvision.transforms.functional as F
#from torchvision.models import resnet50
import torch.nn as nn
from torchvision.models.resnet import resnet50
from lightning.pytorch.callbacks import Callback, ModelCheckpoint

In [2]:
class SimpleSegmentationHead(nn.Sequential):
    def __init__(self, in_channels, num_classes):
        super().__init__(
            nn.Conv2d(in_channels, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, kernel_size=1)
        )

In [3]:
class Identity_2(_Transform):
    """This class is a dummy transform that does nothing. It is useful when
    you want to skip a transform in a pipeline.
    """

    def __call__(self, x: np.ndarray) -> np.ndarray:
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        transform = transforms.Compose([transforms.ToTensor(), normalize])
        return transform(x)

    def __str__(self) -> str:
        return "Identity()"

In [4]:
class Format_label_img(_Transform):

    def __call__(self, x: np.ndarray) -> torch.Tensor:

        if x.ndim == 3:
            x = x.squeeze()

        label_tensor = torch.from_numpy(x).long()

        if label_tensor.min() >= 1:
            label_tensor = label_tensor - 1

        return label_tensor

In [5]:
def model_mIoU(model, trainer, dataloader):
    predictions = trainer.predict(model, dataloader)
    predictions = torch.cat(predictions, dim=0)
    pred_classes = torch.argmax(predictions, dim=1)
    test_samples = np.array([y for _, y in dataloader.test_dataset])
    y = torch.from_numpy(test_samples).long()
    jaccard = JaccardIndex(task="multiclass", num_classes=3)
    score = jaccard(pred_classes, y)
    #print(f"The mIoU of the model is {score.item()*100:.2f}%")
    return score.item()*100

In [6]:
def frag_data_scores_1(model, name, epochs):

    _model = copy.deepcopy(model)

    ## save the results in these folder path
    folder_path = "/frag_dataset_results/"  
    csv_file = os.path.join(folder_path, f"frag_data_{name}.csv")

    with open(csv_file, mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["number_data", "train_loss", "val_loss", "mIoU"])

    for i in range(11,62):
        if i < 10:
            item = "0"+str(i)
        else:
            item = str(i)

        # Path(f'fragmented_data/iteration_{item}/data') is the path to the fragmented MATLAB segmented dataset
        train_fine_data_reader = PNGReader(
            path=Path(f'fragmented_data/iteration_{item}/data')
        )
        train_fine_labels_reader = PNGReader(
            path=Path(f'fragmented_data/iteration_{item}/label')
        )

        train_fine_dataset = SimpleDataset(
            readers=[train_fine_data_reader, train_fine_labels_reader],
            transforms=[Identity_2(), Format_label_img()],
        )

        data_fine_module = MinervaDataModule(
            train_dataset=train_fine_dataset,
            val_dataset=val_fine_dataset,
            test_dataset=test_fine_dataset,
            batch_size=32,
            num_workers=2,
            additional_train_dataloader_kwargs={"drop_last": True},
            name="simulated spectogram"
        )

        trainer_base = L.Trainer(
        max_epochs=epochs,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=False,
        )

        trainer_base.fit(_model, data_fine_module)

        train_loss = _model.train_losses
        val_loss = _model.val_losses
        miou_score = model_mIoU(_model, _trainer, data_fine_module)

        if i == 0:
            number_data = i+1
        else:
            number_data = i*3

        with open(csv_file, mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([number_data, train_loss[epochs-1], val_loss[epochs], miou_score])

        _model = copy.deepcopy(model)

In [7]:
def frag_data_scores_2(model, name, epochs):

    _model = copy.deepcopy(model)

    folder_path = "/frag_dataset_results/"  
    csv_file = os.path.join(folder_path, f"frag_data_{name}_2.csv")

    with open(csv_file, mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["number_data", "train_loss", "val_loss", "mIoU"])

    for i in range(1,11):
        if i < 10:
            item = "0"+str(i)
        else:
            item = str(i)

        train_fine_data_reader = PNGReader(
            path=Path(f'fragmented_data/iteration_{item}/data')
        )
        train_fine_labels_reader = PNGReader(
            path=Path(f'fragmented_data/iteration_{item}/label')
        )

        train_fine_dataset = SimpleDataset(
            readers=[train_fine_data_reader, train_fine_labels_reader],
            transforms=[Identity_2(), Format_label_img()],
        )

        data_fine_module = MinervaDataModule(
            train_dataset=train_fine_dataset,
            val_dataset=val_fine_dataset,
            test_dataset=test_fine_dataset,
            batch_size=32,
            num_workers=2,
            #drop_last=True,
            name="simulated spectogram"
        )

        trainer_base = L.Trainer(
        max_epochs=epochs,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=False,
        )

        trainer_base.fit(_model, data_fine_module)

        train_loss = _model.train_losses
        val_loss = _model.val_losses
        miou_score = model_mIoU(_model, _trainer, data_fine_module)

        if i == 0:
            number_data = i+1
        else:
            number_data = i*3

        with open(csv_file, mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([number_data, train_loss[-1], val_loss[-1], miou_score])

        del(_model)
        _model = copy.deepcopy(model)

In [ ]:
def frag_data_scores_3(model, name, epochs):

    _model = copy.deepcopy(model)

    folder_path = "/frag_dataset_results/"
    csv_file = os.path.join(folder_path, f"frag_data_{name}_3.csv")
    
    with open(f"frag_data_{name}_3.csv", mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["percent", "train_loss", "val_loss", "mIoU"])

    frag = 25
    for i in range(4):
        item = str(frag)

        if frag < 100:
            train_fine_data_reader = PNGReader(
                path=Path(f'fragmented_data_2/{item}_percent/data')
            )
            train_fine_labels_reader = PNGReader(
                path=Path(f'fragmented_data_2/{item}_percent/label')
            )
        else:
            train_fine_data_reader = PNGReader(
            path=Path('dataset_spectogram/train/data')
            )
            train_fine_labels_reader = PNGReader(
                path=Path('dataset_spectogram/train/label')
            )

        train_fine_dataset = SimpleDataset(
        readers=[train_fine_data_reader, train_fine_labels_reader],
        transforms=[Identity_2(), Format_label_img()],
        )
        
        data_fine_module = MinervaDataModule(
            train_dataset=train_fine_dataset,
            val_dataset=val_fine_dataset,
            test_dataset=test_fine_dataset,
            batch_size=32,
            num_workers=0,
            #drop_last=True,
            name="simulated spectogram"
        )
    
        trainer_base = L.Trainer(
        max_epochs=epochs,
        accelerator="gpu",
        devices=1,
        logger=False,
        enable_checkpointing=False,
        )
        
        trainer_base.fit(_model, data_fine_module)
    
        train_loss = _model.train_losses
        val_loss = _model.val_losses
        miou_score = model_mIoU(_model, _trainer, data_fine_module)
        
        with open(csv_file, mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([frag, train_loss[-1], val_loss[-1], miou_score])

        del(_model)
        _model = copy.deepcopy(model)
        frag += 25

In [10]:
val_fine_data_reader = PNGReader(
    path=Path('dataset_spectogram/val/data/')
)

In [11]:
val_fine_labels_reader = PNGReader(
    path=Path('dataset_spectogram/val/label')
)

In [12]:
test_fine_data_reader = PNGReader(
    path=Path('dataset_spectogram/test/data')
)

In [13]:
test_fine_labels_reader = PNGReader(
    path=Path('dataset_spectogram/test/label')
)

In [ ]:
val_fine_dataset = SimpleDataset(
    readers=[val_fine_data_reader, val_fine_labels_reader],
    transforms=[Identity_2(), Format_label_img()],
)

print(val_fine_dataset)

In [ ]:
test_fine_dataset = SimpleDataset(
    readers=[test_fine_data_reader, test_fine_labels_reader],
    transforms=[Identity_2(), Format_label_img()],
)

print(test_fine_dataset)

In [ ]:
_trainer = L.Trainer(
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_checkpointing=False
)

In [17]:
RN50model_default_dilatation = resnet50(weights="DEFAULT",replace_stride_with_dilation=[False, False, True])
RN50model_none_dilatation = resnet50(weights=None,replace_stride_with_dilation=[False, False, True])

In [ ]:
_RN50model_default_dilatation = torch.nn.Sequential(*list(RN50model_default_dilatation.children())[:-2])
model_default_dilatation_simple = DeepLabV3(_RN50model_default_dilatation,SimpleSegmentationHead(2048,3),learning_rate=1e-4,num_classes=3)

In [ ]:
frag_data_scores_1(model_default_dilatation_simple, "resnet50", 50)

In [ ]:
frag_data_scores_2(model_default_dilatation_simple, "resnet50", 50)

In [ ]:
frag_data_scores_3(model_default_dilatation_simple, "resnet50", 50)

In [ ]:
_RN50model_none_dilatation = torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-2])
model_base_simple = DeepLabV3(_RN50model_none_dilatation,SimpleSegmentationHead(2048,3),learning_rate=1e-4,num_classes=3)

In [ ]:
frag_data_scores_1(model_base_simple, "base", 50)

In [ ]:
frag_data_scores_2(model_base_simple, "base", 50)

In [ ]:
frag_data_scores_3(model_base_simple, "base", 50)

# config 1: VerticalLineMask(), AdditiveGaussianNoise(std=60)

## simple seg_head

In [58]:
byol_config_1 = BYOL.load_from_checkpoint("checkpoints_byol/config_1_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_default_dilatation.children())[:-1]), strict=True)
_backbone_config_1 = byol_config_1.backbone

In [59]:
deeplab_backbone_1 = torch.nn.Sequential(*list(_backbone_config_1.children())[:-1])

In [60]:
model_config_1 = DeepLabV3(deeplab_backbone_1,SimpleSegmentationHead(2048,3),learning_rate=1e-4,num_classes=3)

In [ ]:
frag_data_scores_1(model_config_1, "config_1", 50)

In [ ]:
frag_data_scores_2(model_config_1, "config_1", 50)

In [ ]:
frag_data_scores_3(model_config_1, "config_1", 50)

# Config 2: HorizontalLineMask(), ColorJitter()

## simple seg_head

In [20]:
byol_config_2 = BYOL.load_from_checkpoint("checkpoints_byol/config_2_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_2 = byol_config_2.backbone

In [21]:
deeplab_backbone_2 = torch.nn.Sequential(*list(_backbone_config_2.children())[:-1])

In [22]:
model_config_2 = DeepLabV3(deeplab_backbone_2,SimpleSegmentationHead(2048,3),learning_rate=1e-4,num_classes=3)

In [ ]:
frag_data_scores_1(model_config_2, "config_2", 50)

In [ ]:
frag_data_scores_2(model_config_2, "config_2", 50)

In [ ]:
frag_data_scores_3(model_config_2, "config_2", 50)

# Config 3: VerticalLineMask(), ColorJitter()

## simple seg_head

In [24]:
byol_config_3 = BYOL.load_from_checkpoint("checkpoints_byol/config_3_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_3 = byol_config_3.backbone

In [25]:
deeplab_backbone_3 = torch.nn.Sequential(*list(_backbone_config_3.children())[:-1])

In [26]:
model_config_3 = DeepLabV3(deeplab_backbone_3,SimpleSegmentationHead(2048,3),learning_rate=1e-4,num_classes=3)

In [ ]:
frag_data_scores_1(model_config_3, "config_3", 50)

In [ ]:
frag_data_scores_2(model_config_3, "config_3", 50)

In [ ]:
frag_data_scores_3(model_config_3, "config_3", 50)

# Config 4: HorizontalLineMask(), AdditiveGaussianNoise()

## simple seg_head

In [52]:
byol_config_4 = BYOL.load_from_checkpoint("checkpoints_byol/config_4_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_4 = byol_config_4.backbone

In [53]:
deeplab_backbone_4 = torch.nn.Sequential(*list(_backbone_config_4.children())[:-1])

In [54]:
model_config_4 = DeepLabV3(deeplab_backbone_4,SimpleSegmentationHead(2048,3),learning_rate=5e-4,num_classes=3)

In [ ]:
frag_data_scores_1(model_config_4, "config_4", 50)

In [ ]:
frag_data_scores_2(model_config_4, "config_4", 50)

In [ ]:
frag_data_scores_3(model_config_4, "config_4", 50)

# Config 5:
    1 - VerticalLineMask(), ColorJitter()
    2 - HorizontalLineMask(), AdditiveGaussianNoise()

## simple seg_head

In [101]:
byol_config_5 = BYOL.load_from_checkpoint("checkpoints_byol/config_5_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_5 = byol_config_5.backbone

In [102]:
deeplab_backbone_5 = torch.nn.Sequential(*list(_backbone_config_5.children())[:-1])

In [103]:
model_config_5 = DeepLabV3(deeplab_backbone_5,SimpleSegmentationHead(2048,3),learning_rate=1e-4,num_classes=3)

In [ ]:
frag_data_scores_1(model_config_5, "config_5", 50)

In [ ]:
frag_data_scores_2(model_config_5, "config_5", 50)

In [ ]:
frag_data_scores_3(model_config_5, "config_5", 50)

# Config 6:
    1 - VerticalLineMask(), AdditiveGaussianNoise()
    2 - HorizontalLineMask(), ColorJitter()

## simple seg_head

In [62]:
byol_config_6 = BYOL.load_from_checkpoint("checkpoints_byol/config_6_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_6 = byol_config_6.backbone

In [63]:
deeplab_backbone_6 = torch.nn.Sequential(*list(_backbone_config_6.children())[:-1])

In [66]:
model_config_6 = DeepLabV3(deeplab_backbone_6,SimpleSegmentationHead(2048,3),learning_rate=1e-3,num_classes=3)

In [ ]:
frag_data_scores_1(model_config_6, "config_6", 50)

In [ ]:
frag_data_scores_2(model_config_6, "config_6", 50)

In [ ]:
frag_data_scores_3(model_config_6, "config_6", 50)

# config 7:
    1 - VerticalLineMask(), ColorJitter()
    2 - HorizontalLineMask(), ColorJitter()

## simple seg_head

In [39]:
byol_config_7 = BYOL.load_from_checkpoint("checkpoints_byol/config_7_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_7 = byol_config_7.backbone

In [40]:
deeplab_backbone_7 = torch.nn.Sequential(*list(_backbone_config_7.children())[:-1])

In [41]:
model_config_7 = DeepLabV3(deeplab_backbone_7,SimpleSegmentationHead(2048,3),learning_rate=1e-3,num_classes=3)

In [ ]:
frag_data_scores_1(model_config_7, "config_7", 50)

In [ ]:
frag_data_scores_2(model_config_7, "config_7", 50)

In [ ]:
frag_data_scores_3(model_config_7, "config_7", 50)

# config 8:
    1 - VerticalLineMask(), AdditiveGaussianNoise()
    2 - HorizontalLineMask(), AdditiveGaussianNoise()

## simple seg_head

In [97]:
byol_config_8 = BYOL.load_from_checkpoint("checkpoints_byol/config_8_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_8 = byol_config_8.backbone

In [98]:
deeplab_backbone_8 = torch.nn.Sequential(*list(_backbone_config_8.children())[:-1])

In [99]:
model_config_8 = DeepLabV3(deeplab_backbone_8,SimpleSegmentationHead(2048,3),learning_rate=1e-3,num_classes=3)

In [ ]:
frag_data_scores_1(model_config_8, "config_8", 50)

In [ ]:
frag_data_scores_2(model_config_8, "config_8", 50)

In [ ]:
frag_data_scores_3(model_config_8, "config_8", 50)

# No dataset

## no panoradio

In [ ]:
byol_no_panoradio = BYOL.load_from_checkpoint("checkpoints_byol/no_panoradio_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_no_panoradio = byol_no_panoradio.backbone

In [19]:
deeplab_backbone_no_panoradio = torch.nn.Sequential(*list(_backbone_no_panoradio.children())[:-1])

In [18]:
model_no_panoradio = DeepLabV3(deeplab_backbone_no_panoradio,SimpleSegmentationHead(2048,3),learning_rate=1e-4,num_classes=3)

In [ ]:
frag_data_scores_1(model_no_panoradio, "no_panoradio", 50)

In [ ]:
frag_data_scores_2(model_no_panoradio, "no_panoradio", 50)

In [ ]:
frag_data_scores_3(model_no_panoradio, "no_panoradio", 50)

## no powder

In [27]:
byol_no_powder = BYOL.load_from_checkpoint("checkpoints_byol/no_powder_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_no_powder = byol_no_powder.backbone

In [28]:
deeplab_backbone_no_powder = torch.nn.Sequential(*list(_backbone_no_powder.children())[:-1])

In [29]:
model_no_powder = DeepLabV3(deeplab_backbone_no_powder,SimpleSegmentationHead(2048,3),learning_rate=1e-4,num_classes=3)

In [ ]:
frag_data_scores_1(model_no_powder, "no_powder", 50)

In [ ]:
frag_data_scores_1(model_no_powder, "no_powder", 50)

In [ ]:
frag_data_scores_1(model_no_powder, "no_powder", 50)

# Wilcoxon test

In [48]:
from scipy.stats import wilcoxon

In [ ]:
def single_miou_test_img(name):
    miou_wilcoxon = []
    for i in range(5):

        # f"checkpoints_finetuning/{name}_{i}.ckpt" should be the path to the finetuning checkpoints, with the same names saved
        model = DeepLabV3.load_from_checkpoint(f"checkpoints_finetuning/{name}_{i}.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-2]), num_classes=3, strict=True)
        predictions = _trainer.predict(model, data_fine_module)
        predictions = torch.cat(predictions, dim=0)
        pred_classes = torch.argmax(predictions, dim=1)
        test_samples = np.array([y for _, y in data_fine_module.test_dataset])
        y = torch.from_numpy(test_samples).long()
        miou_list = []
        miou_metric = MulticlassJaccardIndex(num_classes=3)
        for i in range(len(y)):
            ious = miou_metric(pred_classes[i].unsqueeze(0), y[i].unsqueeze(0))
            miou_list.append(ious.mean().item()*100)
        miou_wilcoxon = miou_wilcoxon + miou_list
        del(model)

    return miou_wilcoxon

In [35]:
train_fine_data_reader = PNGReader(
    path=Path('dataset_spectogram/train/data')
)
train_fine_labels_reader = PNGReader(
    path=Path('dataset_spectogram/train/label')
)

train_fine_dataset = SimpleDataset(
    readers=[train_fine_data_reader, train_fine_labels_reader],
    transforms=[Identity_2(), Format_label_img()],
)

data_fine_module = MinervaDataModule(
    train_dataset=train_fine_dataset,
    val_dataset=val_fine_dataset,
    test_dataset=test_fine_dataset,
    batch_size=32,
    num_workers=0,
    additional_train_dataloader_kwargs=False,
    name="simulated spectogram"
)

In [39]:
miou_wilcoxon_no_powder = single_miou_test_img("no_powder")

In [41]:
miou_wilcoxon_base = single_miou_test_img("base")

In [ ]:
miou_wilcoxon_resnet = single_miou_test_img("resnet50")

In [ ]:
miou_wilcoxon_config1 = single_miou_test_img("config_1")

In [ ]:
stat, p = wilcoxon(miou_wilcoxon_no_powder, miou_wilcoxon_base)
print("Wilcoxon statistic:", stat)
print("p-value:", p)

In [ ]:
stat, p = wilcoxon(miou_wilcoxon_no_powder, miou_wilcoxon_resnet)
print("Wilcoxon statistic:", stat)
print("p-value:", p)

In [ ]:
stat, p = wilcoxon(miou_wilcoxon_no_powder, miou_wilcoxon_config1)
print("Wilcoxon statistic:", stat)
print("p-value:", p)